# DATA PRE-PROCESSING in GSE69914 - Advanced study of epigenetic mechanisms in the development of neoplasms 

In this notebook, I will evaluate and define the most appropriate **pre-processing operations**.

### Libraries

In [3]:
!pip install methylprep==1.7.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.6 MB/s eta 0:00:0000:01


In [4]:
import methylprep; print(f"✅ methylprep {methylprep.__version__} available")


✅ methylprep 1.7.0 available


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import median_abs_deviation
import warnings
from scipy.stats import mannwhitneyu
from scipy.stats import gaussian_kde
from __future__ import annotations
import os, re, math
from typing import Iterable, Set, Dict, Tuple
from polars import selectors as cs
import polars as pl
from itertools import combinations

%config HistoryManager.enabled = False
warnings.filterwarnings('ignore')


## 1. Import and Data Structure

The processed dataset is imported from the LZ4-compressed **`.parquet`** file generated in the previous step. The structure is already optimized for analysis.

* **File format:** Columnar **`Parquet`** (LZ4 compression) for fast I/O.
* **Rows:** Samples (one per tissue).
* **Columns:**
    * **`id_tissue`**: Unique sample identifier.
    * **`label`**: Numeric class code (**`Int8`**).
    * **`cg`, `ch`**: Methylation probes (**`Float32`**).


In [6]:
# IMPORT DATA STRUCTURE
PARQUET_PATH = "/kaggle/input/gse69914-parquet/GSE69914.parquet"  # CHANGE HERE!! #
ID_COL = "id_tissue"
LABEL_COL = "label"

# Import dataset
GSE69914 = pl.scan_parquet(PARQUET_PATH)
print("✅ Dataset successfully loaded.")


✅ Dataset successfully loaded.


## 2. Data Validation and Integrity Check
* **Dimensions:** The dataset contains **407 samples × 485,514 CpG loci**, matching the expected layout (samples × features).
* **Data types:** Confirmed schema — `id_tissue: String`, `label: Int8`, `probes: Float32`, ensuring compact storage and numerical precision.
* **β-value range:** All values within `[0.000000, 0.997110]`, correctly bounded in [0, 1].
* **Missing values:** No NaN entries detected across any probe — overall missing rate **0%**.

*The methylation matrix is structurally sound, numerically consistent, and fully complete — no cleaning or imputation required before further analysis.*


In [5]:
# DATA VALIDATION CHECK
# Dimensions (rows × columns)
n_rows = GSE69914.select(pl.len().alias("rows")).collect(streaming=True)["rows"][0]
n_cols = len(GSE69914.columns)
print(f"- Dimensions: ({n_rows:,}, {n_cols:,}) → samples × CpG loci")

# Data types
schema = GSE69914.schema
id_dtype = schema.get("id_tissue", None)
label_dtype = schema.get("label", None)

# Detect a few probe columns (cg/ch)
probe_cols = [c for c in schema if c.startswith(("cg", "ch"))][:10]
probe_dtypes = {c: schema[c] for c in probe_cols}

print(f"- Data types:")
print(f"  id_tissue: {id_dtype}")
print(f"  label: {label_dtype}")
print(f"  probes (sample): {list(probe_dtypes.values())[:3]} ...")
print("  ✅ Expected types: (Int8, Float32)")

# Value range check (β-values ∈ [0, 1])
# Compute min and max across all probe columns (efficient aggregation)
probe_exprs = [pl.col(c) for c in schema if c.startswith(("cg", "ch"))]
beta_range = GSE69914.select([
    pl.min_horizontal(probe_exprs).alias("beta_min"),
    pl.max_horizontal(probe_exprs).alias("beta_max")
]).collect(streaming=True)

beta_min = float(beta_range["beta_min"][0])
beta_max = float(beta_range["beta_max"][0])

print(f"- β-value range: [{beta_min:.6f}, {beta_max:.6f}]")
if 0 <= beta_min <= 1 and 0 <= beta_max <= 1:
    print("  ✅ All β-values within expected [0, 1] range.")
else:
    print("  ⚠️  Warning: values outside expected range detected.")


- Dimensions: (407, 485,514) → samples × CpG loci
- Data types:
  id_tissue: String
  label: Int8
  probes (sample): [Float32, Float32, Float32] ...
  ✅ Expected types: (Int8, Float32)
- β-value range: [0.000000, 0.997110]
  ✅ All β-values within expected [0, 1] range.


In [6]:
# Select only numeric CpG/CH probe columns (exclude id_tissue, label)
probe_cols = [c for c in GSE69914.columns if c.startswith(("cg", "ch"))]

# Global NaN count across all probes
null_total = (
    GSE69914.select([pl.col(c).null_count().alias(c) for c in probe_cols])
    .select(pl.sum_horizontal(pl.all()).alias("total_null"))
    .collect(streaming=True)["total_null"][0]
)

# Total number of values (for percentage)
total_values = len(probe_cols) * GSE69914.select(pl.len().alias("rows")).collect(streaming=True)["rows"][0]
missing_rate = null_total / total_values * 100

# Sample check for columns and rows with NaN (to confirm all zero)
col_nulls = (
    GSE69914.select([pl.col(c).null_count().alias(c) for c in probe_cols[:10]])
    .collect(streaming=True)
    .to_dict(as_series=False)
)

print(f"- Total NaN count across all probes: {null_total:,}")
print(f"- Overall missing rate: {missing_rate:.6f}%")

if null_total == 0:
    print("  ✅ No missing entries detected. The methylation matrix is complete.")
else:
    print("  ⚠️ Missing values detected — consider filtering or imputation.")


- Total NaN count across all probes: 0
- Overall missing rate: 0.000000%
  ✅ No missing entries detected. The methylation matrix is complete.


## 3. Technical Filtering
### 3.1 Exclusion of technical probe sets
I excluded probes using curated resources that operationalise known technical artefacts.

* **SNP-affected probes:** probes with common variation at the interrogated CpG, at the single-base extension site, or within the probe body.
* **Cross-reactive probes:** probes with off-target/multi-mapping hybridisation.
* **Design-/platform-specific masks:** consolidated `MASK_*` flags for mapping, SNP windows, non-CpG probes, and optional sex-chromosome probes.
* **Naeem hierarchical QC (450K):** discard logic for multi-mapping, repeats, INDEL, and disruptive SNPs.

**Total CpGs removed: 225,426.**

In [7]:
# EXCLUSION OF TECHNICAL PROBE SETS
# Paths
NAEEM_CSV        = "/kaggle/input/filtering-cpg-naeem/filtering_table.csv"                    # Naeem 2014
PIDSLEY_CSV      = "/kaggle/input/filtering-cpg-pidsley/Chen2013_full_blacklist.csv"          # Pidsley 2016 
CHEN_CROSS_CSV   = "/kaggle/input/filtering-cpg-chen-cross-reactive/chen_2013_cross_reactive.csv"  # Chen 2013 
ZHOU_TSV         = "/kaggle/input/filtering-cpg-zhou/EPIC.anno.GRCh38.tsv"                    # Zhou 2016 
MCCARTNEY_CROSS_CPG    = "/kaggle/input/filtering-cpg-mccartney-2/1-s2.0-S221359601630071X-mmc2.txt"     # McCartney 2016 file 2
MCCARTNEY_CROSS_NONCG  = "/kaggle/input/filtering-cpg-mccartney-3/1-s2.0-S221359601630071X-mmc3.txt" # McCartney 2016 file 3

def load_naeem_drop_set(path: str,
                        probe_cands: Iterable[str]=("probe","IlmnID","ID_REF","cg","Name"),
                        flag_cands:  Iterable[str]=("Flag(discard/keep)","flag(discard/keep)","flag","flag_discard_keep")) -> Set[str]:
    """Return probes flagged as 'discard' in Naeem table."""
    df = pl.read_csv(path)
    probe_col = next((c for c in probe_cands if c in df.columns), None)
    flag_col  = next((c for c in df.columns if str(c).lower().strip().startswith("flag")), None) or next((c for c in flag_cands if c in df.columns), None)
    assert probe_col and flag_col, f"Naeem file columns not found. Seen: {df.columns}"
    probes = df[probe_col].cast(pl.Utf8).str.strip_chars()
    flags  = df[flag_col].cast(pl.Utf8).str.to_lowercase()
    return set(probes.filter(flags.str.contains("discard")).to_list())

def load_pidsley_drop_set(path: str, skip_first_lines: int = 1) -> Set[str]:
    """Robust loader for Pidsley."""
    df_comma = pl.read_csv(path, has_header=True, infer_schema_length=10000,
                           separator=",", skip_rows=skip_first_lines)
    try:
        df_tab = pl.read_csv(path, has_header=True, infer_schema_length=10000,
                             separator="\t", skip_rows=skip_first_lines)
    except Exception:
        df_tab = None
    cand = df_tab if (df_tab is not None and df_tab.width > df_comma.width) else df_comma

    cpg_like_cols = [c for c in cand.columns
                     if ("cg" in str(c).lower()) or ("ilmn" in str(c).lower())
                     or ("probe" in str(c).lower()) or ("id" in str(c).lower())]
    if not cpg_like_cols:
        cpg_like_cols = [cand.columns[0]]

    series_list = cand.select(pl.concat_list([pl.col(c).cast(pl.Utf8) for c in cpg_like_cols]).alias("_u")).to_series()
    s = series_list.explode().cast(pl.Utf8).str.strip_chars()
    values = s.to_list()
    return set(x for x in values if isinstance(x, str) and (x.startswith("cg") or x.startswith("ch")))

def load_firstcol_probe_list(path: str) -> Set[str]:
    """
    Robust 'first column' loader for simple lists.
    Keeps ONLY the first column; returns cg*/ch* IDs.
    """
    def _try_read(sep: str, has_header: bool):
        return pl.read_csv(path, separator=sep, has_header=has_header, infer_schema_length=10000)

    df = None
    for sep in (",", "\t"):
        for hh in (True, False):
            try:
                df = _try_read(sep, hh)
                break
            except Exception:
                df = None
        if df is not None:
            break
    if df is None:  # last resort
        df = pl.read_csv(path, infer_schema_length=10000)

    first_col = df.columns[0]
    s = df[first_col].cast(pl.Utf8).str.strip_chars()
    vals = s.to_list()
    return set(x for x in vals if isinstance(x, str) and (x.startswith("cg") or x.startswith("ch")))

def load_chen_crossreactive_firstcol(path: str) -> Set[str]:
    """Chen 2013 cross-reactive list: CSV; keep ONLY the first column with probe IDs."""
    df = pl.read_csv(path, has_header=True, infer_schema_length=10000, separator=",")
    first_col = df.columns[0]
    s = df[first_col].cast(pl.Utf8).str.strip_chars()
    vals = s.to_list()
    return set(x for x in vals if isinstance(x, str) and (x.startswith("cg") or x.startswith("ch")))

def zhou_mask_ids(tsv_path: str,
                  id_col_candidates: Iterable[str] = ("probeID","IlmnID","Name")) -> Set[str]:
    """Read only ID + MASK_* columns from Zhou TSV/TSV.GZ and return probes to mask."""
    hdr = pl.read_csv(tsv_path, separator="\t", has_header=True, n_rows=1, ignore_errors=True)
    id_col = next((c for c in id_col_candidates if c in hdr.columns), None)
    assert id_col, f"ID column not found in Zhou TSV. Seen: {hdr.columns}"
    mask_cols = [c for c in hdr.columns if str(c).upper().startswith("MASK")]
    assert mask_cols, "No MASK_* columns found in Zhou TSV."

    df = pl.read_csv(tsv_path, separator="\t", has_header=True, columns=[id_col]+mask_cols,
                     dtypes={id_col: pl.Utf8}, ignore_errors=True)

    truthy = {"true","1","t","yes","y"}
    falsy  = {"false","0","f","no","n",""}
    def mask_to_bool(colname: str) -> pl.Expr:
        s = pl.col(colname).cast(pl.Utf8).str.to_lowercase()
        return (pl.when(s.is_in(list(truthy))).then(True)
                  .when(s.is_in(list(falsy))).then(False)
                  .otherwise(False)
                  .alias(colname))
    df_norm = df.select([pl.col(id_col).cast(pl.Utf8).str.strip_chars().alias(id_col)] + [mask_to_bool(c) for c in mask_cols])

    mask_any = pl.any_horizontal([pl.col(c) for c in mask_cols])
    bad = df_norm.filter(mask_any)[id_col].to_list()
    return set(bad)

# Dataset layout
lf = GSE69914.lazy() if isinstance(GSE69914, pl.DataFrame) else GSE69914 
schema_cols = list(lf.schema.keys())
n_cols_before = len(schema_cols)
n_rows_before = lf.select(pl.len()).collect().item()

id_col_candidates = ("IlmnID","probeID","ID_REF","CpG","cg_id","Name")
id_col = next((c for c in id_col_candidates if c in schema_cols), None)

if id_col:
    present_cpgs = set(
        lf.select(pl.col(id_col).cast(pl.Utf8).str.strip_chars().alias("_id")).unique()
          .collect().get_column("_id").to_list()
    )
    layout = "rows"
else:
    cpg_cols = [c for c in schema_cols if isinstance(c, str) and (c.startswith("cg") or c.startswith("ch"))]
    present_cpgs = set(cpg_cols)
    layout = "cols"

# Load lists + intersect with present CpGs
print("Loading technical filter lists...")
S_naeem_all     = load_naeem_drop_set(NAEEM_CSV)
S_pidsley_all   = load_pidsley_drop_set(PIDSLEY_CSV)
S_zhou_all      = zhou_mask_ids(ZHOU_TSV)
S_chenx_all     = load_chen_crossreactive_firstcol(CHEN_CROSS_CSV)

# McCartney = union(File2 CpG-cross, File3 non-CpG-cross) 
S_mccartney2_all = load_firstcol_probe_list(MCCARTNEY_CROSS_CPG)
S_mccartney3_all = load_firstcol_probe_list(MCCARTNEY_CROSS_NONCG)
S_mccartney_all  = S_mccartney2_all | S_mccartney3_all

S_naeem     = S_naeem_all     & present_cpgs
S_pidsley   = S_pidsley_all   & present_cpgs
S_zhou      = S_zhou_all      & present_cpgs
S_chenx     = S_chenx_all     & present_cpgs
S_mccartney = S_mccartney_all & present_cpgs

print(f"• Naeem 2014 (discard flag)                    → in dataset: {len(S_naeem):,}     / total in list: {len(S_naeem_all):,}")
print(f"• Pidsley/Chen (full blacklist)                → in dataset: {len(S_pidsley):,}   / total in list: {len(S_pidsley_all):,}")
print(f"• Zhou 2016 (MASK_*)                           → in dataset: {len(S_zhou):,}      / total in list: {len(S_zhou_all):,}")
print(f"• Chen 2013 (cross-reactive)                   → in dataset: {len(S_chenx):,}     / total in list: {len(S_chenx_all):,}")
print(f"• McCartney 2016 (cross-hyb CpG + non-CpG)     → in dataset: {len(S_mccartney):,} / total in list: {len(S_mccartney_all):,}")

# Union to remove + apply filter 
removed_cpgs = S_naeem | S_pidsley | S_zhou | S_chenx | S_mccartney

if layout == "rows":
    lf_filt = lf.filter(~pl.col(id_col).is_in(list(removed_cpgs)))
else:
    keep_cols = [c for c in schema_cols if c not in removed_cpgs]
    lf_filt = lf.select(keep_cols)

pl.DataFrame({"removed_cpg": sorted(removed_cpgs)}).write_csv("removed_cpgs.csv")
n_cols_after = len(lf_filt.schema)
n_rows_after = lf_filt.select(pl.len()).collect().item()

print("\nSummary:")
print(f"  • Samples × Probes (before): {n_rows_before} × {n_cols_before}")
print(f"  • Samples × Probes (after):  {n_rows_after} × {n_cols_after}")
print(f"  • Total removed CpGs:        {len(removed_cpgs):,}")
print(f"    - from Naeem:      {len(S_naeem):,}")
print(f"    - from Pidsley:    {len(S_pidsley):,}")
print(f"    - from Zhou:       {len(S_zhou):,}")
print(f"    - from Chen-X:     {len(S_chenx):,}")
print(f"    - from McCartney:  {len(S_mccartney):,}")

# Overlaps exclusive partitions
lists: Dict[str, Set[str]] = {
    "Naeem": S_naeem,
    "Pidsley": S_pidsley,
    "Zhou": S_zhou,
    "ChenX": S_chenx,
    "McCartney": S_mccartney,
}
keys = list(lists.keys())

exclusive_counts: Dict[Tuple[str,...], int] = {}
for r in range(1, len(keys)+1):
    for combo in combinations(keys, r):
        inter = set.intersection(*(lists[k] for k in combo))
        others = set().union(*(lists[k] for k in keys if k not in combo))
        excl = inter - others
        exclusive_counts[combo] = len(excl)

print("\nExclusive overlaps (present CpGs only):")
for r in range(1, len(keys)+1):
    for combo in combinations(keys, r):
        lbl = " ∩ ".join(combo)
        print(f"  • {lbl}: {exclusive_counts[combo]:,}")

# save detailed per-CpG source log (5 flags + source count)
pl.DataFrame(
    [{"CpG_ID": cg,
      "From_Naeem":     int(cg in S_naeem),
      "From_Pidsley":   int(cg in S_pidsley),
      "From_Zhou":      int(cg in S_zhou),
      "From_ChenX":     int(cg in S_chenx),
      "From_McCartney": int(cg in S_mccartney)} for cg in removed_cpgs]
).with_columns(
    (pl.col("From_Naeem")+pl.col("From_Pidsley")+pl.col("From_Zhou")+pl.col("From_ChenX")+pl.col("From_McCartney")).alias("Source_Count")
).sort(["Source_Count","CpG_ID"], descending=[True, False])\
 .write_csv("removed_cpgs_log.csv")

print("\nArtifacts:")
print("  • saved removed list → removed_cpgs.csv")
print("  • saved detailed log → removed_cpgs_log.csv")

# Optionally persist the filtered dataset:
GSE69914_filt = lf_filt


Loading technical filter lists...
• Naeem 2014 (discard flag)                    → in dataset: 190,672     / total in list: 190,672
• Pidsley/Chen (full blacklist)                → in dataset: 39,737   / total in list: 67,030
• Zhou 2016 (MASK_*)                           → in dataset: 96,541      / total in list: 193,182
• Chen 2013 (cross-reactive)                   → in dataset: 29,233     / total in list: 29,233
• McCartney 2016 (cross-hyb CpG + non-CpG)     → in dataset: 28,582 / total in list: 44,208

Summary:
  • Samples × Probes (before): 407 × 485514
  • Samples × Probes (after):  407 × 260088
  • Total removed CpGs:        225,426
    - from Naeem:      190,672
    - from Pidsley:    39,737
    - from Zhou:       96,541
    - from Chen-X:     29,233
    - from McCartney:  28,582

Exclusive overlaps (present CpGs only):
  • Naeem: 116,068
  • Pidsley: 987
  • Zhou: 21,787
  • ChenX: 556
  • McCartney: 734
  • Naeem ∩ Pidsley: 1,063
  • Naeem ∩ Zhou: 44,424
  • Naeem ∩ ChenX: 1

In [8]:
n_rows = GSE69914_filt.select(pl.len().alias("rows")).collect(streaming=True)["rows"][0]
n_cols = len(GSE69914_filt.columns)
print(f"- Dimensions: ({n_rows:,}, {n_cols:,}) → samples × CpG loci")


- Dimensions: (407, 260,088) → samples × CpG loci


### 3.2 Annotation-based filtering
ARRIVATA QUA

In [9]:
# === Annotation-based filtering USING Illumina Manifest (CSV with [Heading]/[Assay]) ===
import polars as pl
from typing import Sequence, Set, Dict, List

# --- Inputs ---
# 1) Your beta-matrix AFTER technical lists
lf0 = GSE69914_filt.lazy() if isinstance(GSE69914_filt, pl.DataFrame) else GSE69914_filt

# 2) Illumina official manifest CSV (hg19), e.g.:
#    HumanMethylation450_15017482_v1-2.csv  OR  Infinium MethylationEPIC v1.0 B5 manifest CSV
MANIFEST_CSV = "/kaggle/input/humanmethylation450-15017482-v1-2-manifest/humanmethylation450_15017482_v1-2_manifest.csv"  # <--- adjust

# --- Step A: find the [Assay] header block in the manifest and read the table ---
def _find_assay_skiprows(csv_path: str) -> int:
    """
    Return the number of rows to skip so that the next row is the header line starting with IlmnID,Name,...
    Manifest structure:
      [Heading]
      ... metadata ...
      [Assay]
      IlmnID,Name,AddressA_ID,...
      ...
    """
    skip = 0
    with open(csv_path, "r", encoding="utf-8", errors="ignore") as fh:
        lines = fh.readlines()
    # find the line index of "[Assay]" and then header is the next line
    for i, line in enumerate(lines):
        if line.strip().startswith("[Assay]"):
            skip = i + 1
            break
    return skip

skip_rows = _find_assay_skiprows(MANIFEST_CSV)

# read manifest assay table with Polars
man = pl.read_csv(
    MANIFEST_CSV,
    skip_rows=skip_rows,
    has_header=True,
    infer_schema_length=2000,
    separator=",",
    ignore_errors=True
)

# --- Step B: normalize/validate key columns from the manifest ---
# Column name candidates (Illumina standard):
ID_COL_CANDS   = ("IlmnID", "Name")               # probe ID (IlmnID primary; Name also has cg/ch IDs)
CHR_COL_CANDS  = ("CHR", "Chromosome")
POS_COL_CANDS  = ("MAPINFO", "Coordinate_36", "Coordinate_37", "Coordinate_38")  # MAPINFO is hg19

id_col  = next((c for c in ID_COL_CANDS  if c in man.columns), None)
name_col = "Name" if "Name" in man.columns else None
chr_col = next((c for c in CHR_COL_CANDS if c in man.columns), None)
pos_col = next((c for c in POS_COL_CANDS if c in man.columns), None)
assert id_col is not None, f"Cannot find IlmnID/Name in manifest. Seen: {man.columns}"

man = man.with_columns(
    pl.col(id_col).cast(pl.Utf8).str.strip_chars().alias("_ID")
)

# If Name exists, keep it to detect non-CpG ("ch..."). Otherwise infer from _ID if it already starts with cg/ch.
if name_col:
    man = man.with_columns(pl.col(name_col).cast(pl.Utf8).str.strip_chars().alias("_NAME"))

# CHR validity
if chr_col:
    man = man.with_columns(
        pl.col(chr_col).cast(pl.Utf8).str.strip_chars()
          .str.replace(r"(?i)^chr", "", literal=False)
          .str.to_uppercase().alias("_CHR")
    )
    valid_chr = set([str(i) for i in range(1,23)] + ["X","Y"])
    man = man.with_columns(pl.col("_CHR").is_in(valid_chr).alias("_ok_chr"))
else:
    man = man.with_columns(pl.lit(True).alias("_ok_chr"))

# POS validity
if pos_col:
    man = man.with_columns(pl.col(pos_col).cast(pl.Int64, strict=False).alias("_POS"))
    man = man.with_columns((pl.col("_POS") > 0).fill_null(False).alias("_ok_pos"))
else:
    man = man.with_columns(pl.lit(True).alias("_ok_pos"))

# Detect non-CpG probes (no dedicated column in the manifest; use cg/ch convention):
# - CpG-targeting probes typically have IDs like cg######## (NAME starts with "cg")
# - non-CpG-targeting probes are typically "ch########"
# Prefer NAME if present (more explicit); fallback to _ID.
if name_col:
    man = man.with_columns(
        pl.col("_NAME").str.to_lowercase().str.starts_with("ch").fill_null(False).alias("_is_nonCG")
    )
else:
    man = man.with_columns(
        pl.col("_ID").str.to_lowercase().str.starts_with("ch").fill_null(False).alias("_is_nonCG")
    )

# Duplicate mapping / duplicate IDs in manifest (should be rare, but we guard)
dup_ids = set(
    man.group_by("_ID").len().filter(pl.col("len") > 1).get_column("_ID").to_list()
)
man = man.with_columns(pl.col("_ID").is_in(list(dup_ids)).alias("_is_duplicate"))

# Build fast lookup maps
man_one = man.unique(subset=["_ID"], keep="first")

id_list  = man_one.get_column("_ID").to_list()
ok_chr   = man_one.get_column("_ok_chr").to_list()
ok_pos   = man_one.get_column("_ok_pos").to_list()
is_dup   = man_one.get_column("_is_duplicate").to_list()
is_nonCG = man_one.get_column("_is_nonCG").to_list()

flag_map: Dict[str, Dict[str, bool]] = {
    i: {"ok_chr": c, "ok_pos": p, "dup": d, "nonCG": n}
    for i, c, p, d, n in zip(id_list, ok_chr, ok_pos, is_dup, is_nonCG)
}

# --- Step C: detect dataset layout (CpG rows vs columns) and collect present CpGs ---
schema_cols: Sequence[str] = list(lf0.schema.keys())
id_col_in_df = next((c for c in ("IlmnID","probeID","ID_REF","CpG","cg_id","Name") if c in schema_cols), None)

if id_col_in_df:
    present = set(
        lf0.select(pl.col(id_col_in_df).cast(pl.Utf8).str.strip_chars().alias("_id")).unique()
          .collect().get_column("_id").to_list()
    )
    layout = "rows"
    meta_cols = [c for c in schema_cols if c != id_col_in_df]
else:
    present = set([c for c in schema_cols if isinstance(c, str) and (c.startswith("cg") or c.startswith("ch"))])
    layout = "cols"
    meta_cols = [c for c in schema_cols if c not in present]

# --- Step D: classify and filter with reasons ---
removed_rows: List[Dict[str, str]] = []
keep_ids: Set[str] = set()

for cg in present:
    f = flag_map.get(cg)
    if f is None:
        removed_rows.append({"CpG_ID": cg, "Reason": "not_in_manifest"})
        continue
    reasons = []
    if not f["ok_chr"]:
        reasons.append("invalid_chromosome")
    if not f["ok_pos"]:
        reasons.append("invalid_position")
    if f["dup"]:
        reasons.append("ambiguous_mapping")
    if f["nonCG"]:
        reasons.append("non_CpG_probe")
    if reasons:
        removed_rows.append({"CpG_ID": cg, "Reason": ";".join(reasons)})
    else:
        keep_ids.add(cg)

removed_ids = set(r["CpG_ID"] for r in removed_rows)

# --- Step E: apply to your matrix ---
if layout == "rows":
    lf_annot = lf0.filter(~pl.col(id_col_in_df).cast(pl.Utf8).str.strip_chars().is_in(list(removed_ids)))
else:
    keep_cols = meta_cols + sorted(list(keep_ids if keep_ids else (present - removed_ids)))
    lf_annot = lf0.select(keep_cols)

# --- Step F: artifacts & summary (with reason counts) ---
rem_df = pl.DataFrame(removed_rows)
if rem_df.height == 0:
    print("Manifest cross-check: no probes removed (0).")
else:
    rem_df.write_csv("removed_cpgs_manifest.csv")
    cnt = (
        rem_df
        .with_columns(pl.col("Reason").cast(pl.Utf8).str.split(";").alias("parts"))
        .explode("parts")
        .with_columns(pl.col("parts").cast(pl.Utf8).str.strip_chars().alias("Reason_norm"))
        .filter(pl.col("Reason_norm").is_not_null() & (pl.col("Reason_norm") != ""))
        .group_by("Reason_norm").len()
        .rename({"len": "Count", "Reason_norm": "Reason"})
        .sort("Count", descending=True)
    )
    cnt.write_csv("removed_cpgs_manifest_reasons_counts.csv")
    print("Reasons saved → removed_cpgs_manifest.csv / removed_cpgs_manifest_reasons_counts.csv")
    print(cnt)

n_before = len(present)
n_removed = rem_df.height
print(f"CpGs present at input: {n_before:,}")
print(f"Removed by manifest:  {n_removed:,}")
print(f"Remaining (estimated): {n_before - n_removed:,}")

# Final object for downstream
GSE69914_annot_filt = lf_annot

# (Optional) explicit whitelist size
valid_ids = {i for i, f in flag_map.items() if (f["ok_chr"] and f["ok_pos"] and not f["dup"] and not f["nonCG"])}
print(f"Valid IDs in manifest (hg19): {len(valid_ids):,}")
print("Preview:", list(sorted(valid_ids))[:5])


Reasons saved → removed_cpgs_manifest.csv / removed_cpgs_manifest_reasons_counts.csv
shape: (1, 2)
┌───────────────┬───────┐
│ Reason        ┆ Count │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ non_CpG_probe ┆ 875   │
└───────────────┴───────┘
CpGs present at input: 260,086
Removed by manifest:  875
Remaining (estimated): 259,211
Valid IDs in manifest (hg19): 482,421
Preview: ['cg00000029', 'cg00000108', 'cg00000109', 'cg00000165', 'cg00000236']


In [10]:
n_rows = GSE69914_annot_filt.select(pl.len().alias("rows")).collect(streaming=True)["rows"][0]
n_cols = len(GSE69914_annot_filt.columns)
print(f"- Dimensions: ({n_rows:,}, {n_cols:,}) → samples × CpG loci")


- Dimensions: (407, 259,213) → samples × CpG loci


## 4. Filtering of Invariant CpGs

In [ ]:
# MANCA DA FARE... BISOGNA GUARDARE LE COSE DI NASTARAN CAPIRE SE LE SUE CONCLUSIONI SONO SENSATE O SE NE DOBBIAMO
# TROVARE ALTRE -> IN OGNI CASO CI VA UNA REFERENCES A TUTTO CIòòòòò

## 5. Correction of Infinium I/II Probe Bias

In [ ]:
# QUA BASTA IL PLOT!!

## 6. 